In [1]:
import time
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, datasets
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from pytorch_lightning import Trainer
from torchmetrics.classification import MulticlassAccuracy
import torchvision.models as models

from PIL import Image

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using Device: {DEVICE}")

Using Device: cuda


In [2]:
# JUST OUR USUAL DATASET CLASS
class BrainTumorDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        """Initializes the dataset object. Takes path and transform as inputs"""        
        super().__init__()        
        self.root_dir = root_dir
        self.transform = transform     
        self.classes = sorted(os.listdir(root_dir))        # Manually map folder names to integers
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        
        # Build the file list
        self.images = []
        for cls_name in self.classes:
            cls_path = os.path.join(root_dir, cls_name)
            for img_name in os.listdir(cls_path):
                self.images.append((os.path.join(cls_path, img_name), self.class_to_idx[cls_name]))

    def __len__(self):
        return len(self.images)         # Returns the number of images in the dataset

    def __getitem__(self, idx):
        """Grabs a sepcific image, loads it and applies transforms on it"""
        img_path, label = self.images[idx]
        image = Image.open(img_path).convert("RGB")       
        if self.transform:
            image = self.transform(image)
            
        return image, label


In [3]:
### LOADING THE DATA USING DATALOADER
data_path = '/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset' ### Dataset folder path
## Adding transforms
mri_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.PILToTensor(), ########## CHANGE NO 1 FOR PIXEL PRECISION #########
])

# Instantiate Datasets using the path of the data
train_dataset = BrainTumorDataset(os.path.join(data_path, 'Training'), transform=mri_transforms)
test_dataset = BrainTumorDataset(os.path.join(data_path, 'Testing'), transform=mri_transforms)

# Finally the DataLoader to create and load batches for model training. 
# Note how with a smaller dataset, loading a batch size of 32 is also feasible!
train_loader = DataLoader(
    train_dataset, 
    batch_size=32, 
    shuffle=True, 
    num_workers=os.cpu_count(), 
    pin_memory=True
)
val_loader = DataLoader(
    test_dataset, 
    batch_size=32, 
    shuffle=False, 
    num_workers=os.cpu_count(), 
    pin_memory=True
)
print(f"Loaded {len(train_dataset)} training images across classes: {train_dataset.classes}")

Loaded 5600 training images across classes: ['glioma', 'meningioma', 'notumor', 'pituitary']


In [4]:
class BrainTumorClassifier(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()        
        backbone = models.resnet50(weights="DEFAULT")        
        num_filters = backbone.fc.in_features
        self.feature_extractor = nn.Sequential(*list(backbone.children())[:-1])
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(num_filters, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.feature_extractor(x)
        return self.classifier(x)

# Initialize Model and Optimizer
model = BrainTumorClassifier(num_classes=4).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 230MB/s]


In [5]:
# Native PyTorch AMP Setup
# GradScaler handles dynamic loss scaling to prevent float16 underflow
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))
max_epochs = 40

### Training Loops

One can't escape the training loops on Native PyTorch :) 

In [6]:
start_time=time.time()
for epoch in range(max_epochs):
    # TRAINING LOOP ###############################
    model.train()
    total_train_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for batch_idx, batch in enumerate(train_loader):
        x, y = batch
        x, y = x.to(DEVICE), y.to(DEVICE)
        
        optimizer.zero_grad()
        
        # Runs the forward pass under native autocast context
        with torch.amp.autocast(device_type='cuda', enabled=(DEVICE.type == 'cuda')):
            x = x.to(dtype=torch.float16) / 255.0 ##########CHANGE NO 2 FOR PIXEL PRECISION TYPE CASTING HERE
            logits = model(x)
            loss = F.cross_entropy(logits, y)
            
            # Check if AMP verification is happening - not applicable for devices that run with BF16
            if epoch == 0 and batch_idx == 0 and DEVICE.type == 'cuda':
                print(f"\n[AMP VERIFICATION] Logits tensor data type: {logits.dtype}")
                if logits.dtype == torch.float16:
                    print("[AMP VERIFICATION] Success! Model operations running in float16 mixed precision.\n")
                else:
                    print("[AMP VERIFICATION] Warning: Model operations running in full precision.\n")

        # Scales the loss, performs backpropagation
        scaler.scale(loss).backward()
        
        # Unscales gradients, runs step(), updates scale factor
        scaler.step(optimizer)
        scaler.update()
        
        # Track accuracy and loss manually
        total_train_loss += loss.item() * x.size(0)
        _, predicted = torch.max(logits, 1)
        total_train += y.size(0)
        correct_train += (predicted == y).sum().item()
        
    epoch_train_loss = total_train_loss / len(train_dataset)
    epoch_train_acc = correct_train / total_train
    
    # VALIDATION LOOP ########################
    model.eval()
    total_val_loss = 0.0
    correct_val = 0
    total_val = 0
    
    with torch.no_grad():
        for batch in val_loader:
            x, y = batch
            x, y = x.to(DEVICE), y.to(DEVICE)
            
            # Autocast should also wrap validation inference
            with torch.amp.autocast(device_type='cuda', enabled=(DEVICE.type == 'cuda')):
                x = x.to(dtype=torch.float16) / 255.0 ##### CHANGE NO 3 FOR PIXEL PRECISION
                logits = model(x)
                loss = F.cross_entropy(logits, y)             
            total_val_loss += loss.item() * x.size(0)
            _, predicted = torch.max(logits, 1)
            total_val += y.size(0)
            correct_val += (predicted == y).sum().item()
            
    epoch_val_loss = total_val_loss / len(test_dataset)
    epoch_val_acc = correct_val / total_val
    
    print(f"Epoch {epoch+1}/{max_epochs} -> "
          f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f} | "
          f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f}")

end_time=time.time()
print (f"Time to train: {end_time-start_time:.4f}")


[AMP VERIFICATION] Logits tensor data type: torch.float16
[AMP VERIFICATION] Success! Model operations running in float16 mixed precision.

Epoch 1/40 -> Train Loss: 0.3329 | Train Acc: 0.8877 | Val Loss: 0.6468 | Val Acc: 0.8363
Epoch 2/40 -> Train Loss: 0.1583 | Train Acc: 0.9445 | Val Loss: 0.9678 | Val Acc: 0.7731
Epoch 3/40 -> Train Loss: 0.1314 | Train Acc: 0.9582 | Val Loss: 0.3529 | Val Acc: 0.9163
Epoch 4/40 -> Train Loss: 0.0833 | Train Acc: 0.9714 | Val Loss: 0.8243 | Val Acc: 0.8225
Epoch 5/40 -> Train Loss: 0.0846 | Train Acc: 0.9730 | Val Loss: 0.4127 | Val Acc: 0.9275
Epoch 6/40 -> Train Loss: 0.0600 | Train Acc: 0.9786 | Val Loss: 0.5006 | Val Acc: 0.9156
Epoch 7/40 -> Train Loss: 0.0559 | Train Acc: 0.9832 | Val Loss: 0.4342 | Val Acc: 0.9263
Epoch 8/40 -> Train Loss: 0.0392 | Train Acc: 0.9854 | Val Loss: 0.4796 | Val Acc: 0.9137
Epoch 9/40 -> Train Loss: 0.0382 | Train Acc: 0.9870 | Val Loss: 0.5562 | Val Acc: 0.9200
Epoch 10/40 -> Train Loss: 0.0387 | Train Acc: 0.